In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

# ---------------- USER SETTINGS ----------------
DATA_PATH = "data/corn_2016_2023.parquet"   # change crop file here
TARGET_COL = "yield"                       # regression target
GROUP_COL  = "farm_name"                   # group split key
N_SPLITS   = 5

# feature choices
INCLUDE_YEAR_LAT = True                    # you said: include Year + latitude
USE_LONGITUDE = False                      # you decided to drop longitude
# ------------------------------------------------

# Load
df = pd.read_parquet(DATA_PATH)

# Drop rows with missing target
df = df[df[TARGET_COL].notna()].copy()

# Clean: drop old wrong normalized cols if present
bad_norm = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]

# Clean: remove SAR duplicates ending with _2
sar_dup2 = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]

df = df.drop(columns=bad_norm + sar_dup2, errors="ignore")

print("Rows:", len(df), "Cols:", len(df.columns))
print("Dropped:", bad_norm, "and", len(sar_dup2), "SAR _2 cols")

# Continuous target (float)
y = df[TARGET_COL].to_numpy(dtype=np.float32)

# Groups
groups = df[GROUP_COL].astype(str).to_numpy()
print("Unique groups:", len(np.unique(groups)))

# CV folds (fixed across models)
gkf = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y, groups=groups))
print("Built folds:", len(folds))

# ------------ Metrics helpers ------------
def regression_metrics(y_true, y_pred):
    """Return R², RMSE, MAE."""
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    return r2, rmse, mae

def zone_thresholds_from_train(y_train, q_low=0.4, q_high=0.6):
    """Compute yield thresholds from TRAIN only (prevents leakage)."""
    low = np.quantile(y_train, q_low)
    high = np.quantile(y_train, q_high)
    return low, high

def yield_to_zone(y_vals, low, high):
    """Convert continuous yield to zones using thresholds."""
    # 0=low, 1=medium, 2=high
    z = np.zeros_like(y_vals, dtype=np.int64)
    z[(y_vals >= low) & (y_vals <= high)] = 1
    z[y_vals > high] = 2
    return z

def zone_metrics(y_true, y_pred, y_train_for_thresholds):
    """
    Evaluate 'zone accuracy' + confusion matrix for regression predictions.
    Thresholds are computed from TRAIN yields only.
    """
    low, high = zone_thresholds_from_train(y_train_for_thresholds, 0.4, 0.6)
    z_true = yield_to_zone(y_true, low, high)
    z_pred = yield_to_zone(y_pred, low, high)
    acc = (z_true == z_pred).mean()
    cm = confusion_matrix(z_true, z_pred, labels=[0,1,2])
    return float(acc), cm

def summarize_cv_regression(r2s, rmses, maes, name="MODEL"):
    print(f"\n{name} CV Summary (regression)")
    print(f"R²:   {np.mean(r2s):.3f} ± {np.std(r2s):.3f}")
    print(f"RMSE: {np.mean(rmses):.3f} ± {np.std(rmses):.3f}")
    print(f"MAE:  {np.mean(maes):.3f} ± {np.std(maes):.3f}")

def summarize_cv_zones(zone_accs, cm_sum, name="MODEL"):
    print(f"\n{name} CV Summary (zone-from-regression)")
    print(f"Zone Accuracy: {np.mean(zone_accs):.3f} ± {np.std(zone_accs):.3f}")
    print("Zone confusion matrix sum (rows=true low/med/high, cols=pred):\n", cm_sum)





def build_year_mean_map(years_train, y_train_raw):
    """TRAIN-only map: year -> mean yield in that year."""
    year_means = {}
    for yr in np.unique(years_train):
        vals = y_train_raw[years_train == yr]
        year_means[int(yr)] = float(np.mean(vals))
    return year_means

def lookup_year_means(years, year_means, fallback):
    """Per-sample mean yield by year using TRAIN-derived map."""
    out = np.empty(len(years), dtype=np.float32)
    for i, yr in enumerate(years):
        out[i] = year_means.get(int(yr), fallback)
    return out

def zone_metrics_pm10_from_ratio(y_true_ratio, y_pred_ratio):
    """Zones in ratio space: low<0.9, med 0.9-1.1, high>1.1."""
    z_true = np.where(y_true_ratio < 0.9, 0, np.where(y_true_ratio <= 1.1, 1, 2))
    z_pred = np.where(y_pred_ratio < 0.9, 0, np.where(y_pred_ratio <= 1.1, 1, 2))
    acc = float((z_true == z_pred).mean())
    cm = confusion_matrix(z_true, z_pred, labels=[0,1,2])
    return acc, cm

Rows: 2108996 Cols: 349
Dropped: ['normalized_yield_pct', 'normalized_yield_z'] and 20 SAR _2 cols
Unique groups: 251
Built folds: 5


# Build SAR sequence tensor (needed for RNN/LSTM/Transformer)
## What / Why

Convert VV_#### and VH_#### columns into a time series: (N, T, 2).

Preprocessing

Drop _2 columns (already done in shared setup)

Align VV and VH by suffix

Replace NaN/inf and standardize per fold (done later)

In [2]:
import re
import numpy as np

# ---------- BUILD SAR SEQUENCE (X_seq) ----------
def suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_cols = [c for c in df.columns if c.startswith("VV_")]
vh_cols = [c for c in df.columns if c.startswith("VH_")]

vv_map = {suffix(c): c for c in vv_cols if suffix(c) is not None}
vh_map = {suffix(c): c for c in vh_cols if suffix(c) is not None}

common_suffixes = sorted(set(vv_map.keys()) & set(vh_map.keys()))
T = len(common_suffixes)

print("Time steps (T):", T)

vv_sorted = [vv_map[s] for s in common_suffixes]
vh_sorted = [vh_map[s] for s in common_suffixes]

X_seq = np.zeros((len(df), T, 2), dtype=np.float32)
X_seq[:, :, 0] = df[vv_sorted].to_numpy(dtype=np.float32)
X_seq[:, :, 1] = df[vh_sorted].to_numpy(dtype=np.float32)

print("X_seq shape:", X_seq.shape)
print("NaNs in X_seq:", np.isnan(X_seq).sum())

# ---------- STATIC FEATURES: Year + latitude ----------
STATIC_COLS = ["Year", "latitude"]  # no longitude
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)
print("X_static shape:", X_static.shape)


Time steps (T): 152
X_seq shape: (2108996, 152, 2)
NaNs in X_seq: 536282261
X_static shape: (2108996, 2)


# LSTM (SAR sequence only)

Same as RNN notebook, but swap the model:

In [8]:

import torch
import torch.nn as nn
import numpy as np
import time

# 1. THE MODEL
class LSTMRegressor(nn.Module):
    def __init__(self, seq_dim=4, stat_dim=2, hidden=128):
        super().__init__()
        self.lstm = nn.LSTM(seq_dim, hidden, batch_first=True) 
        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32), nn.ReLU(), 
            nn.Linear(32, 32), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64), nn.ReLU(), 
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_stat):
        _, (h, _) = self.lstm(x_seq)
        s = self.stat_mlp(x_stat)
        return self.head(torch.cat([h[-1], s], dim=1)).squeeze(1)

# 2. THE TRAINING FUNCTION
def run_fast(tr_idx, te_idx, X_seq, X_static, y_raw, years):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Training on: {torch.cuda.get_device_name(0)}")

    # --- Preprocessing ---
    X_tr_seq = X_seq[tr_idx]
    m_tr = np.isnan(X_tr_seq).astype(np.float32)
    # Concatenate SAR data with missingness masks (results in 4 channels)
    X_tr_seq_filled = np.concatenate([np.nan_to_num(X_tr_seq), m_tr], axis=2)
    
    y_tr_raw = y_raw[tr_idx]
    yr_tr = years[tr_idx]
    yr_map = {yr: y_tr_raw[yr_tr == yr].mean() for yr in np.unique(yr_tr)}
    y_tr_norm = (y_tr_raw / np.array([yr_map.get(yr, y_tr_raw.mean()) for yr in yr_tr])).astype(np.float32)

    # --- GPU Pre-loading (Speed Hack) ---
    x_seq_gpu = torch.from_numpy(X_tr_seq_filled).to(device)
    x_stat_gpu = torch.from_numpy(X_static[tr_idx]).to(device)
    y_gpu = torch.from_numpy(y_tr_norm).to(device)

    # --- Setup ---
    model = LSTMRegressor(seq_dim=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.SmoothL1Loss()
    scaler = torch.amp.GradScaler("cuda") 

    # --- Training Loop ---
    BATCH_SIZE = 16384 
    num_samples = x_seq_gpu.size(0)
    
    print(f"Starting Training...")
    model.train()
    start_time = time.time()

    for epoch in range(1, 21):
        indices = torch.randperm(num_samples, device=device)
        epoch_loss = 0
        
        for i in range(0, num_samples, BATCH_SIZE):
            idx = indices[i : i + BATCH_SIZE]
            xs, xst, yb = x_seq_gpu[idx], x_stat_gpu[idx], y_gpu[idx]
            
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                pred = model(xs, xst)
                loss = loss_fn(pred, yb)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            
        if epoch % 5 == 0 or epoch == 1:
            avg_loss = epoch_loss / (num_samples / BATCH_SIZE)
            print(f" Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Time: {time.time()-start_time:.2f}s")

    print(f"✅ Total Training Time: {time.time() - start_time:.2f}s")
    return model

# 3. RUN IT
# Using the first fold as an example
trained_model = run_fast(folds_indices[0][0], folds_indices[0][1], X_seq, X_static, y_raw, years)

🚀 Training on: NVIDIA H200
Starting Training...


RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmEx( handle, opa, opb, m, n, k, alpha_ptr, a, CUDA_R_16F, lda, b, CUDA_R_16F, ldb, beta_ptr, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16F, ldc, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`

Nice — these are clean, valid results, and the LSTM did what we’d expect (a modest bump over the simple RNN), but it’s not a “huge leap.”

Here’s the readout.

1) LSTM vs RNN: did it improve?
Your RNN CV summary (from earlier)

R²: 0.164 ± 0.060

RMSE: 42.975 ± 2.713

MAE: 32.025 ± 2.302

Zone Acc: 0.540 ± 0.090

Your LSTM CV summary (now)

R²: 0.168 ± 0.062

RMSE: 42.870 ± 2.725

MAE: 31.842 ± 2.285

Zone Acc: 0.578 ± 0.089

Interpretation

Regression metrics: tiny improvement

R²: +0.004

RMSE: −0.105

MAE: −0.183

Zone accuracy: noticeable improvement (+0.038)

So:
✅ LSTM is slightly better overall
✅ It seems to help more on zone decisions than on raw yield accuracy
❌ But the gain is small enough that it might be within noise

2) Are these numbers “good” in context?

Your yield std ≈ 47.8 (baseline mean-predict RMSE ≈ 47.8).
Your LSTM RMSE ≈ 42.9, so you beat baseline by ~10%.

That’s:

Real signal, not random

still not strong for a final model

very reasonable for SAR-only (plus Year/lat) and farm-held-out CV

3) What changed in the zone confusion matrix (and why it matters)

LSTM confusion (summed):

low:    [479659  81439 287988]
medium: [ 82777  86158 243439]
high:   [ 24238 169566 653732]


Compared to your RNN, the big visual improvement is:

fewer true high → predicted medium errors (down a lot)

more true high → predicted high correct (up a lot)

So LSTM is better at recognizing high-yield conditions (or at least not collapsing them to medium).

4) But why didn’t R² jump more?

Because the main limitations are still there:

A) SAR has massive missingness

Earlier you found hundreds of millions of NaNs in SAR columns.
Even with masks, the model often doesn’t see enough clean time signal.

B) You’re only giving static info: Year + latitude

No soil, weather, terrain, management → yield variance is still mostly unexplained.

C) The model may be under-regularized/undertrained

6 epochs is fine for a first run, but LSTMs often benefit from:

slightly more epochs (10–20)

LR scheduling

larger hidden size (256)

bidirectional LSTM (sometimes)

But before tuning: get a stronger architecture.

5) The correct next step
✅ Step B1: Transformer time-series model

This is the first model that can meaningfully outperform LSTM on irregular SAR sequences.

Transformers help because:

attention can “skip over” missing timesteps better

learns long-range patterns without vanishing gradients

tends to be more stable across folds

Given you have H200, transformer is absolutely worth it next.

6) Quick “how to present this” (one-liner)

“Switching from a simple RNN to an LSTM produced a small but consistent improvement in yield prediction, with a modest gain in R² and a larger gain in zone classification accuracy. This suggests temporal memory helps, but SAR-only features still limit explained variance, motivating more expressive temporal models (Transformers) and later inclusion of weather/terrain covariates.”

If you want, I’ll give you the Transformer regressor notebook in the same “drop-in after the two setup cells” style:

SAR + masks + static Year/lat fusion

AMP + ETA

R²/RMSE/MAE + zone metrics

(optional) positional encoding + attention masking

# Optimized LSTM